In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
#files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*blos_202505*'))
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/202604_/blos/*'))
files

['/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260409T014503_V202608281152C_0644090501.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260409T080003_V202608281152C_0644090503.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260409T110003_V202608281152C_0644090504.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260409T140003_V202608281153C_0644090505.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260409T170003_V202608281153C_0644090506.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260409T200003_V202608281154C_0644090507.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260409T230003_V202608281154C_0644090508.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260410T014503_V202608281154C_0644100501.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos_20260410T050003_V202608281155C_0644100502.fits',
 '/home/ulyanov/data/solo/phi/202604_/blos/phi-fdt-blos

In [3]:
data1, alpha1 = make_map(files, correct_dr=True, sine_lat=True, minmu=0.1, delta_lon=180, pow=4)

/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:118: RuntimeWarning: invalid value encountered in divide
  mean_alpha += np.nan_to_num((alpha - mean_alpha) * weight / coverage)


In [15]:
data2, alpha2 = make_map(files, correct_dr=True, sine_lat=True, minmu=0.1, delta_lon=180, pow=1)

/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:118: RuntimeWarning: invalid value encountered in divide
  mean_alpha += np.nan_to_num((alpha - mean_alpha) * weight / coverage)


In [45]:
show_map(data1, vmin=-1000, vmax=1000, sine_lat=True)

In [77]:
nx = len(data1)
#lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

flux1 = np.nanmean(data1, axis=1)
flux2 = np.nanmean(data2, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux1)
plt.plot(lat, flux2)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_179232/964258859.py:5: RuntimeWarning: Mean of empty slice
  flux1 = np.nanmean(data1, axis=1)
/tmp/ipykernel_179232/964258859.py:6: RuntimeWarning: Mean of empty slice
  flux2 = np.nanmean(data2, axis=1)


In [67]:
from scipy.ndimage import gaussian_filter

w = (alpha2 - alpha1) / (alpha1 + alpha2)
delta = (data2 - data1) / (alpha1 + alpha2) * w / (w ** 2 + 0.05 ** 2)
#delta = gaussian_filter(np.nan_to_num(delta), 20)


show_map(delta, cmap='seismic', vmin=-30, vmax=30, sine_lat=True)

In [46]:
show_map(w, cmap='gray', vmin=0, vmax=0.2, sine_lat=True)

In [63]:
show_map(data1 - delta * alpha1, vmin=-1000, vmax=1000, sine_lat=True)

In [78]:
nx = len(data1)
#lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

flux1_ = np.nanmean(data1 - delta * alpha1, axis=1)
flux2_ = np.nanmean(data2 - delta * alpha2, axis=1)
delta_ = np.nanmean(delta, axis=1)


plt.figure(figsize=(10,8))
plt.plot(lat, delta_)
plt.plot(lat, flux1_)
plt.plot(lat, flux2_)


plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_179232/1283037189.py:5: RuntimeWarning: Mean of empty slice
  flux1_ = np.nanmean(data1 - delta * alpha1, axis=1)
/tmp/ipykernel_179232/1283037189.py:6: RuntimeWarning: Mean of empty slice
  flux2_ = np.nanmean(data2 - delta * alpha2, axis=1)
/tmp/ipykernel_179232/1283037189.py:7: RuntimeWarning: Mean of empty slice
  delta_ = np.nanmean(delta, axis=1)


In [74]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2310.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
header['T_OBS']

'2026.04.29_07:08:43_TAI'

In [75]:
show_map(data_, vmin=-1000, vmax=1000, sine_lat=True)

In [81]:
temp = data_.copy()
flux_ = np.nanmean(temp, axis=1)

nx, ny = data_.shape
lat_ = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_, flux_)
plt.plot(lat, flux2)
plt.plot(lat, flux2_)


plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_179232/1940017635.py:2: RuntimeWarning: Mean of empty slice
  flux_ = np.nanmean(temp, axis=1)
